# MartiniSurf Colab Workflow: Protein Surface Systems (AutoMartini M3)

This notebook is organized for reproducible scientific runs with a fixed execution sequence.

**Execution flow:**
1. Install environment and dependencies
2. (Optional) Generate linker from SMILES
3. Configure and run the full system build
4. Download outputs
5. Visualize final immobilized structure


## Step 1. Environment Setup
Install MartiniSurf and required tooling before running any downstream cell.


In [ ]:
#@title Step 1 • Install MartiniSurf (supports private repo via token)
REPO_URL = "https://github.com/jjimenezgar/MartiniSurf.git" #@param {type:"string"}
PRIVATE_REPO = True #@param {type:"boolean"}
INSTALL_VIEWER = True #@param {type:"boolean"}
ENSURE_MARTINIZE2 = True #@param {type:"boolean"}
ENSURE_DSSP_TOOL = False #@param {type:"boolean"}

import getpass
import shlex
import shutil
import subprocess
from pathlib import Path


def run_cmd(cmd, check=True):
    if isinstance(cmd, str):
        res = subprocess.run(cmd, shell=True, text=True, capture_output=True)
        shown = cmd
    else:
        res = subprocess.run(cmd, text=True, capture_output=True)
        shown = ' '.join(shlex.quote(x) for x in cmd)
    if check and res.returncode != 0:
        raise RuntimeError(f"Command failed ({shown})\\nSTDOUT:\\n{res.stdout[-4000:]}\\nSTDERR:\\n{res.stderr[-4000:]}")
    return res

run_cmd('rm -rf /content/MartiniSurf', check=False)

if PRIVATE_REPO:
    token = getpass.getpass('Paste your GitHub token (repo read access): ').strip()
    if not token:
        raise RuntimeError('Token is required for private repository installation.')
    if REPO_URL.startswith('https://'):
        clone_url = REPO_URL.replace('https://', f'https://{token}@', 1)
    else:
        raise RuntimeError('REPO_URL must use https://')
else:
    clone_url = REPO_URL

run_cmd(['git', 'clone', clone_url, '/content/MartiniSurf'])
run_cmd(['python', '-m', 'pip', 'install', '-q', '--upgrade', 'pip'])
run_cmd(['python', '-m', 'pip', 'install', '-q', '-e', '/content/MartiniSurf'])

if ENSURE_MARTINIZE2 and shutil.which('martinize2') is None:
    attempts = [
        ['python', '-m', 'pip', 'install', '-q', 'martinize2'],
        ['python', '-m', 'pip', 'install', '-q', 'vermouth'],
    ]
    for cmd in attempts:
        run_cmd(cmd, check=False)
        if shutil.which('martinize2') is not None:
            break

if ENSURE_DSSP_TOOL:
    # 🔹 Install DSSP (mkdssp) in Colab
    import subprocess
    import shutil

    print("🔧 Updating package list...")
    subprocess.run(["apt-get", "update", "-qq"], check=False)

    print("🔧 Installing DSSP...")
    subprocess.run(["apt-get", "install", "-y", "-qq", "dssp"], check=True)

    print("\n🔍 Checking installation...")
    path = shutil.which("mkdssp")

    if path:
        print(f"✅ mkdssp found at: {path}")
        subprocess.run(["mkdssp", "--version"], check=False)
    else:
        raise RuntimeError("❌ mkdssp was not installed correctly.")

if INSTALL_VIEWER:
    run_cmd(['python', '-m', 'pip', 'install', '-q', 'py3Dmol'])

print('Installed.')
print('martinisurf:', shutil.which('martinisurf') or 'NOT FOUND')
print('martinize2:', shutil.which('martinize2') or 'NOT FOUND')
print('mkdssp:', shutil.which('mkdssp') or 'NOT FOUND')


## Step 2. Optional Linker Preparation
Generate a linker candidate only if your workflow uses linker mode.


In [ ]:
#@title Step 2 • Optional Linker Generation (AutoMartini M3)
RUN_LINKER_GENERATION = True  #@param {type:"boolean"}
INPUT_MODE = "SMILES"  #@param ["SMILES", "Upload SMILES file"]
SMILES = "CCCCCCCCC"  #@param {type:"string"}
MOLECULE_NAME = "LNK"  #@param {type:"string"}

import subprocess
from pathlib import Path

# Ensure RDKit is available
subprocess.run(["pip", "install", "-q", "rdkit"], check=False)

if RUN_LINKER_GENERATION:

    print("🔹 AutoMartini M3 Linker Generation\n")

    # --- Handle upload mode ---
    if INPUT_MODE == "Upload SMILES file":
        from google.colab import files
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError("No file uploaded.")

        filename = next(iter(uploaded.keys()))
        filepath = Path("/content") / filename

        content = filepath.read_text().strip().split()
        if not content:
            raise RuntimeError("Uploaded file is empty.")

        SMILES = content[0]
        print(f"📎 Loaded SMILES from file: {SMILES}\n")

    if not SMILES.strip():
        raise RuntimeError("SMILES string is empty.")

    if not MOLECULE_NAME.strip():
        raise RuntimeError("Molecule name is empty.")

    # --- Minimal deps ---
    subprocess.run(
        ["pip", "install", "-q", "numpy", "scipy", "networkx"],
        check=True
    )

    # --- Clone repo if needed ---
    if not Path("/content/Automartini_M3").exists():
        subprocess.run(
            ["git", "clone",
             "https://github.com/Martini-Force-Field-Initiative/Automartini_M3"],
            check=True
        )

    # --- Install AutoMartini as editable package ---
    subprocess.run(
        ["pip", "install", "-e", "/content/Automartini_M3"],
        check=True
    )

    # --- Execute AutoMartini ---
    cmd = [
        "python",
        "-m",
        "auto_martiniM3",
        "--mode", "run",
        "--smi", SMILES,
        "--mol", MOLECULE_NAME
    ]

    print(f"⚙️ Running AutoMartini M3 for '{MOLECULE_NAME}' ...\n")
    run = subprocess.run(cmd, capture_output=True, text=True)

    if run.returncode != 0:
        print(run.stderr)
        raise RuntimeError("AutoMartini M3 failed.")

    # --- Check output ---
    itp_file = Path(f"/content/{MOLECULE_NAME}.itp")
    gro_file = Path(f"/content/{MOLECULE_NAME}.gro")

    if itp_file.exists() and gro_file.exists():
        print("✅ Linker successfully generated!")
        print(f"   - {itp_file.name}")
        print(f"   - {gro_file.name}")
    else:
        print("⚠️ Files not found in /content")
        subprocess.run(["ls", "-lh", "/content"])

else:
    print("⏭️ Linker generation skipped.")


## Step 3. System Build Configuration
Define scientific inputs and run the pipeline. Keep beginner settings unless advanced control is required.


In [ ]:
#@title Step 3 • Build Protein System (Beginner + Advanced Panels)
#@markdown Recommended flow: set **Beginner options** first, then expand advanced controls only when needed.

# --------------------------------------------------
# Beginner Panel (recommended defaults)
# --------------------------------------------------
PDB_INPUT_MODE = "PDB ID" #@param ["PDB ID", "Upload PDB file"]
PDB_ID = "1RJW" #@param {type:"string"}
ORIENTATION_MODE = "Anchor mode" #@param ["Anchor mode", "Linker mode"]
USE_EXISTING_SURFACE = False #@param {type:"boolean"}
USE_DSSP = True #@param {type:"boolean"}
ANCHOR_GROUPS = "1 8 10 11; 2 1025 1027 1028" #@param {type:"string"}
LINKER_GROUPS = "1 8 10 11; 2 1025 1027 1028" #@param {type:"string"}
DIST = 10.0 #@param {type:"number"}
OUTDIR = "Simulation_Files" #@param {type:"string"}
DRY_RUN = False #@param {type:"boolean"}

# --------------------------------------------------
# Advanced Panel (expert tuning)
# --------------------------------------------------
SHOW_ADVANCED_OPTIONS = False #@param {type:"boolean"}

# Martinization controls
ADV_MOLTYPE = "Protein" #@param {type:"string"}
ADV_USE_GO_MODEL = False #@param {type:"boolean"}
ADV_MARTINIZE_MAXWARN = 1 #@param {type:"integer"}
ADV_MERGE_GROUPS = "A,B,C,D" #@param {type:"string"}

# Surface controls
ADV_SURFACE_SOURCE = "Upload surface.gro now" #@param ["Upload surface.gro now", "Use existing surface.gro in session"]
ADV_SURFACE_FILE = "surface.gro" #@param {type:"string"}
ADV_SURFACE_TOPOLOGY = "4-1" #@param ["4-1", "2-1"]
ADV_SURFACE_DX_MODE = "Preset by topology" #@param ["Preset by topology", "Custom"]
ADV_SURFACE_DX_NM = 0.27 #@param {type:"number"}
ADV_LX = 20.0 #@param {type:"number"}
ADV_LY = 20.0 #@param {type:"number"}
ADV_SURFACE_BEAD = "C1" #@param {type:"string"}

# Linker controls
ADV_LINKER_SOURCE = "Upload linker files now" #@param ["Upload linker files now", "Use existing linker.gro/.itp in session"]
ADV_LINKER_GRO = "linker.gro" #@param {type:"string"}
ADV_INVERT_LINKER = False #@param {type:"boolean"}
ADV_SURFACE_LINKERS = 0 #@param {type:"integer"}

import importlib.util
import shlex
import shutil
import subprocess
from pathlib import Path

# Resolve effective settings from basic + advanced panels.
if SHOW_ADVANCED_OPTIONS:
    MOLTYPE = ADV_MOLTYPE
    USE_GO_MODEL = ADV_USE_GO_MODEL
    MARTINIZE_MAXWARN = int(ADV_MARTINIZE_MAXWARN)
    MERGE_GROUPS = ADV_MERGE_GROUPS

    SURFACE_SOURCE = ADV_SURFACE_SOURCE
    SURFACE_FILE = ADV_SURFACE_FILE
    SURFACE_TOPOLOGY = ADV_SURFACE_TOPOLOGY
    SURFACE_DX_MODE = ADV_SURFACE_DX_MODE
    SURFACE_DX_NM = float(ADV_SURFACE_DX_NM)
    LX = float(ADV_LX)
    LY = float(ADV_LY)
    SURFACE_BEAD = ADV_SURFACE_BEAD

    LINKER_SOURCE = ADV_LINKER_SOURCE
    LINKER_GRO = ADV_LINKER_GRO
    INVERT_LINKER = ADV_INVERT_LINKER
    SURFACE_LINKERS = int(ADV_SURFACE_LINKERS)
else:
    MOLTYPE = "Protein"
    USE_GO_MODEL = False
    MARTINIZE_MAXWARN = 1
    MERGE_GROUPS = "A,B,C,D"

    SURFACE_SOURCE = "Upload surface.gro now"
    SURFACE_FILE = "surface.gro"
    SURFACE_TOPOLOGY = "4-1"
    SURFACE_DX_MODE = "Preset by topology"
    SURFACE_DX_NM = 0.27
    LX = 20.0
    LY = 20.0
    SURFACE_BEAD = "C1"

    LINKER_SOURCE = "Upload linker files now"
    LINKER_GRO = "linker.gro"
    INVERT_LINKER = False
    SURFACE_LINKERS = 0


def parse_group_lines(raw_text, field_name):
    groups = []
    for ln in (raw_text or '').replace(';', '\n').splitlines():
        line = ln.strip()
        if not line:
            continue
        parts = line.replace(',', ' ').split()
        if len(parts) < 2:
            raise ValueError(f'{field_name}: invalid line "{line}". Use: GROUP RESID [RESID ...]')
        groups.append(parts)
    return groups


def parse_merge_groups(raw_text):
    vals = []
    for token in (raw_text or '').replace(';', '\n').splitlines():
        item = token.strip()
        if item:
            vals.append(item)
    return vals

if shutil.which('martinisurf') is None:
    raise RuntimeError('martinisurf is not installed. Run the Install cell first.')
if shutil.which('martinize2') is None:
    raise RuntimeError('martinize2 is not installed. Re-run Install with ENSURE_MARTINIZE2=True.')

if USE_DSSP:
    has_mdtraj = importlib.util.find_spec('mdtraj') is not None
    has_mkdssp = shutil.which('mkdssp') is not None
    if not has_mdtraj and not has_mkdssp:
        print('Warning: USE_DSSP=True but mdtraj/mkdssp are not detected in PATH. MartiniSurf will auto-try available DSSP options.')

if PDB_INPUT_MODE == 'Upload PDB file':
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError('No PDB uploaded')
    pdb_input = '/content/' + next(iter(uploaded.keys()))
else:
    pdb_input = PDB_ID.strip()

if USE_EXISTING_SURFACE:
    print('Using existing surface file from session/upload.')
    if SURFACE_SOURCE == 'Upload surface.gro now':
        from google.colab import files
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError('No surface file uploaded')
        surface_path = '/content/' + next(iter(uploaded.keys()))
    else:
        surface_path = '/content/' + SURFACE_FILE
        if not Path(surface_path).exists():
            raise FileNotFoundError(f'Missing file: {surface_path}')

cmd = ['martinisurf', '--pdb', pdb_input, '--moltype', MOLTYPE, '--outdir', OUTDIR, '--maxwarn', str(int(MARTINIZE_MAXWARN))]
if USE_GO_MODEL:
    cmd += ['--go']
if not USE_DSSP:
    cmd += ['--no-dssp']

for mg in parse_merge_groups(MERGE_GROUPS):
    cmd += ['--merge', mg]

if USE_EXISTING_SURFACE:
    print('Using existing surface file from session/upload.')
    cmd += ['--surface', surface_path]
else:
    preset_dx = 0.27 if SURFACE_TOPOLOGY == '4-1' else 0.47
    if SURFACE_DX_MODE == 'Preset by topology':
        dx_nm = preset_dx
    else:
        dx_nm = float(SURFACE_DX_NM)
    if dx_nm <= 0:
        raise ValueError('SURFACE_DX_NM must be > 0.')
    print(f'Generated surface preset: {SURFACE_TOPOLOGY} (dx={dx_nm} nm, mode={SURFACE_DX_MODE})')
    cmd += ['--surface-mode', SURFACE_TOPOLOGY, '--lx', str(LX), '--ly', str(LY), '--dx', str(dx_nm), '--surface-bead', SURFACE_BEAD]

if ORIENTATION_MODE == 'Linker mode':
    if LINKER_SOURCE == 'Upload linker files now':
        from google.colab import files
        print('Upload linker.gro and linker.itp (matching basename preferred)')
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError('No linker files uploaded')
        names = list(uploaded.keys())
        gros = [n for n in names if n.lower().endswith('.gro')]
        itps = [n for n in names if n.lower().endswith('.itp')]
        if not gros:
            raise RuntimeError('Please upload one linker .gro file')
        if not itps:
            raise RuntimeError('Please upload one linker .itp file')
        linker_gro = Path('/content') / gros[0]
        linker_itp = Path('/content') / itps[0]
        target_itp = linker_gro.with_suffix('.itp')
        if linker_itp != target_itp:
            shutil.copyfile(linker_itp, target_itp)
        linker_path = str(linker_gro)
    else:
        linker_path = '/content/' + LINKER_GRO
        if not Path(linker_path).exists():
            raise FileNotFoundError(f'Missing linker file: {linker_path}')
        expected_itp = Path(linker_path).with_suffix('.itp')
        if not expected_itp.exists():
            raise FileNotFoundError(f'Missing linker topology: {expected_itp}')

    cmd += ['--linker', linker_path]
    if INVERT_LINKER:
        cmd += ['--invert-linker']
    if int(SURFACE_LINKERS) > 0:
        cmd += ['--surface-linkers', str(int(SURFACE_LINKERS))]

    groups = parse_group_lines(LINKER_GROUPS, 'LINKER_GROUPS')
    if not groups:
        raise ValueError('LINKER_GROUPS is empty. Add at least one group line.')
    for g in groups:
        cmd += ['--linker-group'] + g
else:
    groups = parse_group_lines(ANCHOR_GROUPS, 'ANCHOR_GROUPS')
    if not groups:
        raise ValueError('ANCHOR_GROUPS is empty. Add at least one group line.')
    for g in groups:
        cmd += ['--anchor'] + g
    cmd += ['--dist', str(DIST)]

print('Running:\n' + ' '.join(shlex.quote(x) for x in cmd))
if DRY_RUN:
    print('DRY_RUN=True, command not executed.')
else:
    res = subprocess.run(cmd, text=True, capture_output=True)
    print(res.stdout[-12000:])
    if res.returncode != 0:
        print(res.stderr[-12000:])
        combined = ((res.stdout or '') + '\n' + (res.stderr or '')).lower()
        msg = 'MartiniSurf failed. '
        if 'warnings were encountered after accounting for the -maxwarn flag' in combined:
            msg += 'Increase MARTINIZE_MAXWARN (for example 1 -> 3).'
        elif 'dssp' in combined and ('failed' in combined or 'not found' in combined):
            msg += 'Likely cause: DSSP setup issue. Keep USE_DSSP enabled with mdtraj installed, or disable USE_DSSP.'
        elif 'martinize2' in combined and ('not found' in combined or 'no such file' in combined):
            msg += 'Likely cause: martinize2 missing in PATH.'
        elif 'failed to download rcsb' in combined:
            msg += 'Likely cause: network issue or invalid PDB ID.'
        raise RuntimeError(msg)

    print('Build completed')


## Step 4. Export Results
Package simulation outputs for archival and downstream analysis.


In [ ]:
#@title Step 4 • Download Results as ZIP
OUTPUT_FOLDER = "Simulation_Files" #@param {type:"string"}
ZIP_NAME = "Simulation_Files" #@param {type:"string"}

import shutil
from pathlib import Path
from google.colab import files

folder = Path('/content') / OUTPUT_FOLDER
if not folder.exists():
    raise FileNotFoundError(f'Folder not found: {folder}')

shutil.make_archive(ZIP_NAME, 'zip', str(folder))
files.download(ZIP_NAME + '.zip')


## Step 5. Visual Quality Check
Inspect the generated immobilized system geometry before simulation.


In [ ]:
#@title Step 5 • Viewer (Generated Structure)
SYSTEM_GRO_FILE = "Simulation_Files/2_system/immobilized_system.gro" #@param {type:"string"}
STYLE = "Spheres" #@param ["Spheres", "Sticks"]

import py3Dmol
from pathlib import Path

path = Path('/content') / SYSTEM_GRO_FILE
if not path.exists():
    raise FileNotFoundError(f'File not found: {path}')

model = path.read_text()
view = py3Dmol.view(width='100%', height=800)
view.addModel(model, 'gro')
if STYLE == 'Sticks':
    view.setStyle({'stick': {}})
else:
    view.setStyle({'sphere': {'radius': 1.4}})
view.zoomTo()
view.show()


## Acknowledgements, Licenses, and Citations
This notebook uses external tools and libraries with their own licenses and citation requirements.

### Please cite
- **Martinize2 / Vermouth**: Kroon PC, Grünewald F, Barnoud J, et al. *Martinize2 and Vermouth: Unified Framework for Topology Generation*. eLife (reviewed preprint v2). DOI: `10.7554/eLife.90627.2`.
- **AutoMartini M3**: Szczuka M, Pereira GP, Walter LJ, Gueroult M, Poulain P, Bereau T, Souza PCT, Chavent M. *Fast Parametrization of Martini3 Models for Fragments and Small Molecules*. Journal of Chemical Theory and Computation, 2025, 22(1):610-623.

### Licenses
- **MartiniSurf**: this repository license.
- **martinize2 / Vermouth**: see upstream project license.
- **AutoMartini M3**: see upstream project license.
- Python packages used here (depending on workflow): `numpy`, `scipy`, `MDAnalysis`, `mdtraj`, `py3Dmol`.
